In [ ]:
# Cell 1 — Imports and path setup
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # only physical GPU 1 visible (shows as cuda:0)

import sys

FMRG_DIR = os.path.expanduser("~/flow_matching/fmrg")
sys.path.insert(0, FMRG_DIR)

import torch
from PIL import Image
import matplotlib.pyplot as plt

print(f"Visible GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Using: {torch.cuda.get_device_name(0)}")



In [ ]:
# Cell 1b — HuggingFace authentication (required for gated FLUX.1-dev)
# Paste your HF token below (read-only scope is enough).
# Get one at: https://huggingface.co/settings/tokens
# You also need to accept the license at:
#   https://huggingface.co/black-forest-labs/FLUX.1-dev

import os
from huggingface_hub import login

# Disable Xet Storage backend — avoids "Background writer channel closed" crash
os.environ["HF_HUB_DISABLE_XET"] = "1"

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError(
        "Set HF_TOKEN env var before running this notebook.\n"
        "  export HF_TOKEN=hf_..."
    )
login(token=HF_TOKEN)
print("Logged in to HuggingFace.")

In [ ]:
# Cell 2 — Load FLUX FlowMap model
# ─────────────────────────────────────────────────────────────────────────────
# Memory (bfloat16)          Quantized memory
#   Transformer  ~12 GB      int8  ~6 GB   int4/nf4 ~3 GB
#   VAE          ~0.5 GB
#   Activations  ~1-2 GB
# ─────────────────────────────────────────────────────────────────────────────
# QUANTIZE_MODE options:
#   "none"  – full bfloat16, best quality   (~12 GB transformer)
#   "int8"  – torchao int8 weight-only       (~6 GB, near-lossless)
#   "int4"  – torchao int4 weight-only       (~3 GB, minor degradation)
#   "nf4"   – bitsandbytes NF4 4-bit         (~3 GB, best 4-bit quality)
# ─────────────────────────────────────────────────────────────────────────────

import importlib
import fluxfm_sampler_reward
importlib.reload(fluxfm_sampler_reward)
from fluxfm_sampler_reward import FluxFlowMapSampler

PROMPT = "photo of a woman"
LORA_PATH = os.path.join(FMRG_DIR, "checkpoints", "flux-flowmap-lora")

QUANTIZE_MODE = "int8"   # change to "none" / "int4" / "nf4" as needed
VAE_ON_CPU    = False     # move VAE to CPU to save another ~0.5 GB (slower decode)

sampler = FluxFlowMapSampler(
    model_id="black-forest-labs/FLUX.1-dev",
    lora_path=LORA_PATH,
    device="cuda",
    dtype=torch.bfloat16,
    prompt=PROMPT,
    quantize_mode=QUANTIZE_MODE,
)

if VAE_ON_CPU:
    sampler.vae.to("cpu")
    sampler.pipeline.vae.to("cpu")


In [ ]:
# Cell 3 — Unguided sampling (no reward)
# sample_forward: plain flow-ODE, num_steps=4 at 256x256.

NUM_STEPS = 5

for mode in ["flow_matching", "flow_map1", "flow_map2"]:

    image_pil, latent, noise, intermediates = sampler.sample_simple(
        num_steps=NUM_STEPS,
        height=256,
        width=256,
        guidance_scale=3.5,
        seed=42,
        return_intermediates=True,
        sample_mode=mode,
    )

    # Intermediate steps: step 0 = noise, steps 1..N = after each solver step
    n_cols = len(intermediates)
    fig, axes = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 3.5))
    if n_cols == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, intermediates)):
        ax.imshow(img)
        ax.axis("off")
        ax.set_title("noise" if i == 0 else f"step {i}/{NUM_STEPS}", fontsize=10)
    fig.suptitle(f"Unguided sampling  |  {PROMPT[:50]}", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()

    # Final image (same as last intermediate)
    plt.figure(figsize=(5, 5))
    plt.imshow(image_pil)
    plt.axis("off")
    plt.title(f"Final  |  prompt: {PROMPT[:60]}")
    plt.tight_layout()
    plt.show()

    image_pil.save("unguided_sample.png")
    print("Saved unguided_sample.png")

In [ ]:
import gc
from image_mpc import run_image_mpc_demo, make_flux_wrappers, receding_horizon_flow_control, fmrg_j_image, trajectory_rewards, tune_sampler_speed
from fluxfm_sampler_reward import CLIPReward, create_clip_preprocess, HF_CACHE
from transformers import AutoProcessor


def gpu_mem_gb() -> float:
    return torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0


def free_gpu(*names):
    """Delete named locals and release cached CUDA memory."""
    frame = globals()
    for name in names:
        if name in frame:
            del frame[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Drop leftovers from a previous run of this cell
free_gpu(
    "traj_unc", "traj_mpc", "controls_mpc", "z0", "z_unc", "z_mpc",
    "clip_reward", "clip_tokenizer", "txt_features", "prompt_tokens",
)

device = next(sampler.pipeline.transformer.parameters()).device
dtype  = next(sampler.pipeline.transformer.parameters()).dtype

prompt         = "blonde hair person"
guidance_scale = 3.5
n_total_steps    = 5
seed           = 42
height         = 256
width          = 256
h_horizon      = 4
lam            = 1000.0
n_inner        = 5
lr_inner       = 1e-2
SAMPLE_MODE    = "flow_map2"   # "flow_map2" | "flow_map1" | "flow_matching"

# tune_sampler_speed(sampler, fast=True)

for control_type in ['fmrg', 'mpc']:   # 'mpc' | 'fmrg'

    v_theta_mpc, _, endpoint_map = make_flux_wrappers(
        sampler, prompt,
        guidance_scale=guidance_scale,
        n_mpc_steps=n_total_steps,
        sample_mode=SAMPLE_MODE,
    )

    # CLIP reward: cosine similarity between decoded image and prompt (higher = better)
    print("Loading CLIP reward model...")
    clip_tokenizer = AutoProcessor.from_pretrained(
        "laion/CLIP-ViT-H-14-laion2B-s32B-b79K",
        cache_dir=HF_CACHE,
    )
    clip_reward = CLIPReward(
        device=device, dtype=torch.float32, tokenizer=clip_tokenizer, cache_dir=HF_CACHE,
    )
    clip_preprocess = create_clip_preprocess(224)

    with torch.no_grad():
        prompt_tokens = clip_tokenizer(
            prompt, return_tensors="pt", padding=True, max_length=77, truncation=True
        ).to(device)
        txt_features = clip_reward.model.get_text_features(**prompt_tokens)
        txt_features = txt_features / txt_features.norm(dim=-1, keepdim=True)


    def r_image(z: torch.Tensor) -> torch.Tensor:
        """CLIP cosine similarity between decoded image and prompt. Returns (B,)."""
        img = sampler.decode(z)
        img_01 = ((img + 1.0) / 2.0).float()
        preprocessed = clip_preprocess(img_01)
        img_features = clip_reward.model.get_image_features(preprocessed)
        img_features = img_features / img_features.norm(dim=-1, keepdim=True)
        return (img_features @ txt_features.T).squeeze(-1)


    def eval_reward(z: torch.Tensor) -> torch.Tensor:
        """Evaluate r on CPU or GPU latents."""
        with torch.no_grad():
            return r_image(z.to(device=device, dtype=dtype))


    # Initial noise latent
    torch.manual_seed(seed)
    in_ch   = sampler.pipeline.transformer.config.in_channels // 4
    vscale  = sampler.vae_scale_factor
    H_lat, W_lat = height // vscale, width // vscale
    z0 = torch.randn(1, in_ch, H_lat, W_lat, device=device, dtype=dtype)

    # Uncontrolled baseline (same sample_mode)
    dt_flux = 1.0 / n_total_steps
    print(f"Running uncontrolled baseline ({SAMPLE_MODE})...")
    with torch.no_grad():
        z_unc = z0.clone()
        traj_unc_list = [z_unc.clone()]
        ts_unc = torch.linspace(1.0, 0.0, n_total_steps + 1, device=device, dtype=dtype)
        for j in range(n_total_steps):
            t_c = ts_unc[j].item()
            if SAMPLE_MODE == "flow_map2":
                t_n = max(0.0, t_c - dt_flux)
            elif SAMPLE_MODE == "flow_map1":
                t_n = 0.0
            else:
                t_n = t_c
            dt_unc = (ts_unc[j] - ts_unc[j + 1]).item()
            u = sampler.predict_vector(
                z_unc, t_c,
                prompt_embeds=sampler.cached_prompt_embeds,
                pooled_prompt_embeds=sampler.cached_pooled_prompt_embeds,
                guidance_scale=guidance_scale,
                t_next=t_n,
            )
            z_unc = z_unc - dt_unc * u
            traj_unc_list.append(z_unc.clone())
    traj_unc = torch.stack(traj_unc_list, dim=0)
    free_gpu("traj_unc_list", "ts_unc", "u")
    print(f"  after uncontrolled rollout: {gpu_mem_gb():.1f} GB")

    img_unc = sampler.pipeline.image_processor.postprocess(
        sampler.decode_no_grad(z_unc).float().detach().cpu(), output_type="pil"
    )[0]
    free_gpu("z_unc")
    print(f"  after uncontrolled decode: {gpu_mem_gb():.1f} GB")

    # MPC-controlled run
    print(f"Running {control_type} with ({SAMPLE_MODE})...")
    if control_type == 'mpc':
        z_control, traj_control, u, mu_final = receding_horizon_flow_control(
            v_theta   = v_theta_mpc,
            r         = r_image,
            x0        = z0,
            N         = n_total_steps,
            H         = h_horizon,
            lam       = lam,
            optimizer = "adam",
            n_iters   = n_inner,
            lr        = lr_inner,
            verbose   = True,
            device    = device,
            warmstart = True,
            drift_detach = True,
        )
        
    elif control_type == 'fmrg':
        z_control, traj_control, u = fmrg_j_image(
            sampler, r_image, z0,
            N=n_total_steps, n_opt=n_inner, lam=lam,
            guidance_scale=guidance_scale,
            sample_mode=SAMPLE_MODE,
            return_traj=True, return_controls=True,
            verbose=True,
        )

    free_gpu("z0")
    print(f"  after running {control_type}: {gpu_mem_gb():.1f} GB")

    img_control = sampler.pipeline.image_processor.postprocess(
        sampler.decode_no_grad(z_control).float().detach().cpu(), output_type="pil"
    )[0]
    free_gpu("z_control")
    print(f"  after {control_type} decode: {gpu_mem_gb():.1f} GB")

    # Side-by-side image plot
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_unc); axes[0].axis("off"); axes[0].set_title(f"Uncontrolled ({SAMPLE_MODE})")
    axes[1].imshow(img_control); axes[1].axis("off")
    axes[1].set_title(f"{control_type}  H={h_horizon}  λ={lam}  N_inner={n_inner}  ({SAMPLE_MODE})")
    plt.suptitle(prompt[:70], fontsize=10)
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    free_gpu()

    # Reward vs time (MPC progress s: 0=noise, 1=data)
    t_grid = torch.linspace(0.0, 1.0, traj_unc.shape[0]).cpu().float().numpy()
    r_unc     = trajectory_rewards(traj_unc, r_image, device=device, dtype=dtype).float().numpy()
    r_control = trajectory_rewards(traj_control, r_image, device=device, dtype=dtype).float().numpy()
    free_gpu("traj_unc", "traj_control")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(t_grid, r_unc, "o-", label="Uncontrolled", linewidth=2, markersize=6)
    ax.plot(t_grid, r_control, "s-", label=control_type, linewidth=2, markersize=6)
    ax.set_xlabel("Progress time $s$  (0 = noise, 1 = data)")
    ax.set_ylabel("CLIP score")
    ax.set_title("CLIP similarity along trajectory")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    print(f"\nFinal CLIP  uncontrolled: {float(r_unc[-1]):.4f}")
    print(f"Final CLIP  {control_type}:          {float(r_control[-1]):.4f}")
    print(f"Max CLIP    uncontrolled: {float(r_unc.max()):.4f}")
    print(f"Max CLIP    {control_type}:          {float(r_control.max()):.4f}")

    ctrl_norms = u.flatten(2).norm(dim=2).cpu()
    print(f"\nMean control ‖u‖ per step: {ctrl_norms.mean(dim=1).tolist()}")

    # Unload CLIP and other run-time-only objects
    clip_reward.cpu()
    free_gpu(
        "clip_reward", "clip_tokenizer", "clip_preprocess",
        "txt_features", "prompt_tokens", "controls_mpc", "mu_final",
        "v_theta_mpc", "endpoint_map",
    )
    print(f"GPU mem after cleanup: {gpu_mem_gb():.1f} GB")


In [ ]:
# Cell 4 — Reward-guided sampling (FMRG-J, optional)
# Requires reward models (HPSv2, ImageReward, CLIP, PickScore).
# Comment out if you only want the unguided sample above.

from fluxfm_sampler_reward import RewardEnsemble

reward = RewardEnsemble(
    device="cuda",
    dtype=torch.float32,
    hps_weight=5.0,
    imagereward_weight=1.0,
    pickscore_weight=0.05,
    clip_weight=0.01,
)

img_tensor = sampler.sample_reward_guided(
    reward_model=reward,
    num_steps=6,
    guidance_scale=3.5,
    step_size=3.0,
    img_shape=(256, 256),
    grad_mode="jac",          # FMRG-J
    sample_mode="flow_map1",  # 1 NFE/step
    num_optim_iters=3,
    early_stop=4,
    seed=42,
)

img_np = ((img_tensor[0].float().permute(1, 2, 0).cpu().clamp(0, 1)) * 255).byte().numpy()
img_guided = Image.fromarray(img_np.squeeze() if img_np.ndim == 4 else img_np)

plt.figure(figsize=(5, 5))
plt.imshow(img_guided)
plt.axis("off")
plt.title(f"FMRG-J guided  |  {PROMPT[:60]}")
plt.tight_layout()
plt.show()
img_guided.save("fmrg_j_sample.png")
print("Saved fmrg_j_sample.png")